System construction and test


In [1]:
from datetime import date, datetime
import pandas as pd
import yfinance as yf
import time
import numpy as np
#from Alert import Alert
pd.options.mode.chained_assignment = None  # default='warn'
#alert = Alert('1h','GGAL')

input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)
Parametros a variar para back testing ( definir rangos de variação)
     k, d, smooth (parametros do stch)
     dayM  (frequencia media) (multiplicador da frequencia mais alta) exemplo: 5
     semM  (frequencia baixa) (multiplicador da frequencia media) exemplo: 7

In [4]:
dataini = '2018-04-03 19:30:00'
datafim = '2023-04-03 19:30:00'

In [6]:
#%%timeit
import sqlite3
import pandas as pd

# Caminho para o banco de dados
caminho_bd = r'C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\Alpha_Vantage\sqtitulosalpha.db'

# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)

# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}'
ORDER BY datetime
"""

dftitulosdados = pd.read_sql_query(consulta, conexao)

# Fechando a conexão
conexao.close()

# Exibindo os primeiros registros para conferir
#display(dftitulosdados)
dftitulosdados = dftitulosdados.drop(columns=["symbol", "moeda", "intervalo","volume"])
#display(len(dftitulosdados))
#display(dftitulosdados.head(10))

In [46]:

i = 'high'
K = 16  
D = 5    
smoth = 5  
medM = 8  
lowM = 10
stpl = 0.02
comission = 0.0035
taxalivrerisgoprom = 0.05


In [50]:
#%%timeit
# Stochastic calculation
def stochastic(df, i, K, D, smoth):
        
    df["k"] = (100. * (df.close - df.low.rolling(K).min()) /
        (df.high.rolling(K).max() - df.low.rolling(K).min()))
    
    df["k" + i ] = df.k.rolling(smoth).mean()
    df["d" + i ] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)  

    return df
dfstoch = stochastic(dftitulosdados, i, K, D, smoth)

display (dfstoch.head(10))
#%time dfstoch

,datetime,open,high,low,close,khigh,dhigh
0,2018-04-04 09:00:00,51.1569,51.8740,50.6553,50.6631,NaN,NaN
1,2018-04-04 10:00:00,50.6396,51.9328,50.6317,51.6507,NaN,NaN
2,2018-04-04 11:00:00,51.6250,51.8936,51.4900,51.6585,NaN,NaN
3,2018-04-04 12:00:00,51.6624,51.7408,51.4469,51.6036,NaN,NaN
4,2018-04-04 13:00:00,51.6036,51.6820,51.5527,51.6271,NaN,NaN
5,2018-04-04 14:00:00,51.6193,52.0034,51.5919,51.9015,NaN,NaN
6,2018-04-04 15:00:00,51.9015,52.1209,51.8858,52.0817,NaN,NaN
7,2018-04-04 16:00:00,52.0974,52.0974,52.0974,52.0974,NaN,NaN
8,2018-04-05 09:00:00,52.1209,52.8067,51.6507,52.5520,NaN,NaN
9,2018-04-05 10:00:00,52.7793,52.8028,52.0112,52.1209,NaN,NaN


In [52]:
# stochastic high, med and low frcuency

def stoch_hml( df, k, d, smth, medM, lowM): 
    
    df = stochastic(df, "high", k, d, smth)
    df = stochastic(df, "med", k*medM, d*medM, smth*medM)
    df = stochastic(df, "low", k*medM*lowM, d*medM*lowM, smth*medM*lowM)
    
    return df
    



In [54]:
dfstoch_hml= stoch_hml(dfstoch , K, D, smoth, medM , lowM )
display(dfstoch_hml.head(10) )

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow
0,2018-04-04 09:00:00,51.1569,51.8740,50.6553,50.6631,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-04-04 10:00:00,50.6396,51.9328,50.6317,51.6507,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-04-04 11:00:00,51.6250,51.8936,51.4900,51.6585,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-04-04 12:00:00,51.6624,51.7408,51.4469,51.6036,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-04-04 13:00:00,51.6036,51.6820,51.5527,51.6271,NaN,NaN,NaN,NaN,NaN,NaN
5,2018-04-04 14:00:00,51.6193,52.0034,51.5919,51.9015,NaN,NaN,NaN,NaN,NaN,NaN
6,2018-04-04 15:00:00,51.9015,52.1209,51.8858,52.0817,NaN,NaN,NaN,NaN,NaN,NaN
7,2018-04-04 16:00:00,52.0974,52.0974,52.0974,52.0974,NaN,NaN,NaN,NaN,NaN,NaN
8,2018-04-05 09:00:00,52.1209,52.8067,51.6507,52.5520,NaN,NaN,NaN,NaN,NaN,NaN
9,2018-04-05 10:00:00,52.7793,52.8028,52.0112,52.1209,NaN,NaN,NaN,NaN,NaN,NaN


In [58]:
def system_criterias (df):   # input  df() =  dfstoch_hml ()

    df["longbuylow"] = ((df["klow"] > 20) & (df["klow"] > df["dlow"])).astype(int)
    df["longbuymed"] = ((df["kmed"] > 20) & (df["kmed"] > df["dmed"])).astype(int)
    df["longbuyhigh"] = ((df["khigh"] > 20) & (df["khigh"] > df["dhigh"])).astype(int)

    return df


In [62]:
dfcriterias = system_criterias (dfstoch_hml)
display (dfcriterias.head(10))

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow,longbuylow,longbuymed,longbuyhigh
0,2018-04-04 09:00:00,51.1569,51.8740,50.6553,50.6631,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
1,2018-04-04 10:00:00,50.6396,51.9328,50.6317,51.6507,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,2018-04-04 11:00:00,51.6250,51.8936,51.4900,51.6585,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
3,2018-04-04 12:00:00,51.6624,51.7408,51.4469,51.6036,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,2018-04-04 13:00:00,51.6036,51.6820,51.5527,51.6271,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
5,2018-04-04 14:00:00,51.6193,52.0034,51.5919,51.9015,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
6,2018-04-04 15:00:00,51.9015,52.1209,51.8858,52.0817,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
7,2018-04-04 16:00:00,52.0974,52.0974,52.0974,52.0974,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
8,2018-04-05 09:00:00,52.1209,52.8067,51.6507,52.5520,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
9,2018-04-05 10:00:00,52.7793,52.8028,52.0112,52.1209,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


In [64]:
#%%timeit
def system_signals(df):   #input dfcriterias
   
    
    df["state"] = "standby"    
    
    
    for i in range(1, len(df)):        

        # long buy  states          

        if  df.loc[i, "longbuyhigh"] == 1 and df.loc[i, "longbuymed"] == 1 and df.loc[i, "longbuylow"] == 1 and df.loc[i-1, "state"] == "standby" :        
            df.loc[i, "state"] = "buylong"

            
            
        if  df.loc[i, "longbuymed"] == 1 and df.loc[i, "longbuylow"] == 1 and (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong") :
            df.loc[i, "state"] = "staylong" 

            
        if  df.loc[i, "longbuyhigh"] == 1 and df.loc[i, "longbuylow"] == 1 and df.loc[i, "longbuymed"] == 0 and (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong") :
            df.loc[i, "state"] = "staylong"

        # long sell state and price 
        
        if  (df.loc[i, "longbuylow"] == 0 ) and  (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong")  :
             df.loc[i, "state"] = "selllong"
            
            
        if  df.loc[i, "longbuyhigh"] == 0 and df.loc[i, "longbuymed"] == 0 and  (df.loc[i-1, "state"] == "buylong" or df.loc[i-1, "state"] == "staylong") :
             df.loc[i, "state"] = "selllong"
             
            
        if  (df.loc[i, "longbuylow"] == 0 or df.loc[i, "longbuymed"] == 0) and  df.loc[i-1, "state"] == "selllong"  :
             df.loc[i, "state"] = "standby" 
      
    return df  
 

In [66]:
dfsignals = system_signals(dfcriterias)
display (dfsignals.head(10))

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow,longbuylow,longbuymed,longbuyhigh,state
0,2018-04-04 09:00:00,51.1569,51.8740,50.6553,50.6631,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
1,2018-04-04 10:00:00,50.6396,51.9328,50.6317,51.6507,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
2,2018-04-04 11:00:00,51.6250,51.8936,51.4900,51.6585,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
3,2018-04-04 12:00:00,51.6624,51.7408,51.4469,51.6036,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
4,2018-04-04 13:00:00,51.6036,51.6820,51.5527,51.6271,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
5,2018-04-04 14:00:00,51.6193,52.0034,51.5919,51.9015,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
6,2018-04-04 15:00:00,51.9015,52.1209,51.8858,52.0817,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
7,2018-04-04 16:00:00,52.0974,52.0974,52.0974,52.0974,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
8,2018-04-05 09:00:00,52.1209,52.8067,51.6507,52.5520,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
9,2018-04-05 10:00:00,52.7793,52.8028,52.0112,52.1209,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby


In [68]:
#%%timeit
def system_signals_opt1 (df):
    n = len(df)
    state_array = np.full(n, "standby", dtype=object)  # inicializa com "standby"
    estado_anterior = "standby"

    high = df["longbuyhigh"].to_numpy()
    med = df["longbuymed"].to_numpy()
    low = df["longbuylow"].to_numpy()

    for i in range(1, n):
        if high[i] == 1 and med[i] == 1 and low[i] == 1 and estado_anterior == "standby":
            state_array[i] = "buylong"
            estado_anterior = "buylong"
        elif med[i] == 1 and low[i] == 1 and estado_anterior in ["buylong", "staylong"]:
            state_array[i] = "staylong"
            estado_anterior = "staylong"
        elif high[i] == 1 and low[i] == 1 and med[i] == 0 and estado_anterior in ["buylong", "staylong"]:
            state_array[i] = "staylong"
            estado_anterior = "staylong"
        elif low[i] == 0 and estado_anterior in ["buylong", "staylong"]:
            state_array[i] = "selllong"
            estado_anterior = "selllong"
        elif high[i] == 0 and med[i] == 0 and estado_anterior in ["buylong", "staylong"]:
            state_array[i] = "selllong"
            estado_anterior = "selllong"
        elif (low[i] == 0 or med[i] == 0) and estado_anterior == "selllong":
            state_array[i] = "standby"
            estado_anterior = "standby"
        else:
            state_array[i] = estado_anterior

    df["state"] = state_array
    return df

In [70]:
dfsignals = system_signals_opt1(dfcriterias)
display (dfsignals.head(10))

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow,longbuylow,longbuymed,longbuyhigh,state
0,2018-04-04 09:00:00,51.1569,51.8740,50.6553,50.6631,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
1,2018-04-04 10:00:00,50.6396,51.9328,50.6317,51.6507,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
2,2018-04-04 11:00:00,51.6250,51.8936,51.4900,51.6585,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
3,2018-04-04 12:00:00,51.6624,51.7408,51.4469,51.6036,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
4,2018-04-04 13:00:00,51.6036,51.6820,51.5527,51.6271,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
5,2018-04-04 14:00:00,51.6193,52.0034,51.5919,51.9015,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
6,2018-04-04 15:00:00,51.9015,52.1209,51.8858,52.0817,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
7,2018-04-04 16:00:00,52.0974,52.0974,52.0974,52.0974,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
8,2018-04-05 09:00:00,52.1209,52.8067,51.6507,52.5520,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
9,2018-04-05 10:00:00,52.7793,52.8028,52.0112,52.1209,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby


dfstate = long_buy_crits_numpy(dfstoch_hml)
pd.set_option('display.max_rows', None)
dfbuysellop = dfstate[(dfstate['state'] == 'buylong') | (dfstate['state'] == 'selllong')]
#df_intervalo = dfstate.iloc[800:1500] 
#display(df_intervalo)
display (len(dfbuysell))
display(dfbuysell.head(10))

Stop Loss Reentry

In [73]:
#%%timeit
def Stop_Loss_Reentry (dfsignals, stpl) :

    df = dfsignals[(dfsignals['state'] == 'buylong') | (dfsignals['state'] == 'staylong')]
    df = df.reset_index(drop=True)
    df = df.drop(columns=["open" ,	"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
    #drop(columns=["symbol", "moeda", "intervalo","volume"])
    df["stpl"] = 0.0
    stoplossprice = 0.0
    lastlongbuyprice = 0.0

    for i in range(0, len(df)):
        
        if df.loc[i, "state"] == "buylong" :
           stoplossprice = df.loc[i, "close"]
           df.loc[i,"stpl"] = df.loc[i, "close"] - stoplossprice * (1 - stpl)
           
        if df.loc[i, "state"]== "staylong" :
           df.loc[i, "stpl"] = df.loc[i, "close"] - stoplossprice * (1- stpl)
            
           if df.loc[i, "stpl"] < 0.0 :
                df.loc[i, "state"] = "selllong"
               
           if df.loc[i, "stpl"] > 0.0 and   (df.loc[i-1, "state"] == "selllong" or df.loc[i-1, "state"] == "sellstay") :
                df.loc[i, "state"] = "buylong"
               
           if df.loc[i, "stpl"] < 0.0 and   (df.loc[i-1, "state"] == "selllong" or df.loc[i-1, "state"] == "sellstay") :
                df.loc[i, "state"] = "sellstay"

    return df


In [75]:

dfstoploss = Stop_Loss_Reentry (dfsignals, stpl)
display(dfstoploss.head(10))

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,stpl
0,2019-04-03 09:00:00,19.7475,1,1,1,buylong,0.39495
1,2019-04-03 10:00:00,19.5580,1,1,1,staylong,0.20545
2,2019-04-03 11:00:00,19.3053,1,1,1,selllong,-0.04725
3,2019-04-03 12:00:00,19.2737,1,1,1,sellstay,-0.07885
4,2019-04-03 13:00:00,19.3778,1,1,1,buylong,0.02525
5,2019-04-03 14:00:00,19.2895,1,1,1,selllong,-0.06305
6,2019-04-03 15:00:00,19.2579,1,1,0,sellstay,-0.09465
7,2019-04-03 16:00:00,19.2579,1,1,0,sellstay,-0.09465
8,2019-04-04 09:00:00,19.4474,1,1,1,buylong,0.09485
9,2019-04-04 10:00:00,19.5896,1,1,1,staylong,0.23705


In [59]:
dflongsell = dfstate.drop(columns=["open" ,	"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
dflongsell = dflongsell[(dflongsell['state'] == 'selllong')]
display (dflongsell.head(10))
# 
dfbuysellstpl = dflongstpl[(dflongstpl['state'] == 'buylong') | (dflongstpl['state'] == 'selllong')]
df_intervalo = dfbuysellstpl.iloc[0:100]
#display(len(dfbuysell))
#display(df_intervalo)


,datetime,close,khora,dhora,kdia,ddia,ksem,dsem,longbuylow,longbuymed,longbuyhigh,state
2113,2019-04-08 09:00:00,20.0001,64.266937,67.639763,22.790971,22.882963,60.147706,52.895325,1,0,0,selllong
2159,2019-04-15 14:00:00,20.9042,88.880491,87.434039,34.248962,24.784463,55.801589,55.871668,0,1,1,selllong
2603,2019-07-01 14:00:00,28.3296,63.355680,75.924050,88.452492,88.573569,47.299618,36.762951,1,0,0,selllong
2650,2019-07-10 11:00:00,30.0882,57.984963,75.521858,86.180598,86.214648,56.878060,37.635612,1,0,0,selllong
2674,2019-07-15 11:00:00,29.3847,21.600706,29.585273,84.785556,85.399040,60.967531,38.807664,1,0,0,selllong
2798,2019-08-05 13:00:00,27.0548,32.264110,33.385376,56.689797,57.788569,75.912049,49.521956,1,0,0,selllong
4848,2020-07-01 09:00:00,7.7139,16.450868,14.553077,42.190843,42.276119,30.646136,17.805250,1,0,0,selllong
4934,2020-07-14 16:00:00,9.0089,19.215362,15.022711,76.410302,76.613012,36.154306,22.880055,1,0,0,selllong
4951,2020-07-16 16:00:00,9.2007,72.134101,75.108414,77.019016,77.941999,37.222068,23.964567,1,0,0,selllong
5035,2020-07-29 11:00:00,9.7443,17.632588,12.990049,79.613932,80.555405,42.789075,29.371571,1,0,0,selllong


In [61]:
#%%timeit
# Suponha que df1 e df2 têm as mesmas colunas (incluindo 'datetime')
dflongbuysell = pd.concat([dflongsell, dfbuysellstpl], ignore_index=True)

# Ordenar pelo datetime (certifique-se de que é do tipo datetime)
dflongbuysell["datetime"] = pd.to_datetime(dflongbuysell["datetime"])
dflongbuysell = dflongbuysell.sort_values("datetime").reset_index(drop=True)
display (dflongbuysell.head(10))

,datetime,close,khora,dhora,kdia,ddia,ksem,dsem,longbuylow,longbuymed,longbuyhigh,state,stpl
0,2019-04-03 09:00:00,19.7475,24.679281,18.247179,22.486269,22.446346,61.922141,50.880358,1,1,1,buylong,0.394950
1,2019-04-03 11:00:00,19.3053,34.827139,25.162081,23.217566,22.057614,61.803550,51.058875,1,1,1,selllong,-0.047250
2,2019-04-03 13:00:00,19.3778,43.836951,34.595241,23.860475,21.799215,61.688073,51.235772,1,1,1,buylong,0.025250
3,2019-04-03 14:00:00,19.2895,40.722729,37.803931,24.050418,21.718477,61.625637,51.323597,1,1,1,selllong,-0.063050
4,2019-04-04 09:00:00,19.4474,43.912940,41.277722,24.197150,21.638435,61.406292,51.584288,1,1,1,buylong,0.094850
5,2019-04-04 11:00:00,19.3132,52.803921,44.910678,24.423349,21.711956,61.253023,51.755556,1,1,1,selllong,-0.039350
6,2019-04-04 14:00:00,19.3763,53.643746,52.573663,24.036467,21.932488,61.006954,52.008513,1,1,1,buylong,0.023750
7,2019-04-08 09:00:00,20.0001,64.266937,67.639763,22.790971,22.882963,60.147706,52.895325,1,0,0,selllong,NaN
8,2019-04-10 14:00:00,20.0791,83.734096,67.846403,22.852051,22.833477,58.392710,54.409336,1,1,1,buylong,0.401582
9,2019-04-11 10:00:00,19.6646,78.112525,82.130267,23.897042,22.775972,58.021715,54.669704,1,1,0,selllong,-0.012918


In [63]:
#%%timeit
df = dflongbuysell

# Supondo que seu DataFrame se chame df
cond = (df["state"] == "selllong") & (df["stpl"].isna()) & (df["state"].shift(1) == "selllong")

dflongbuysell = df[~cond].reset_index(drop=True)
display (dflongbuysell.head(10))

,datetime,close,khora,dhora,kdia,ddia,ksem,dsem,longbuylow,longbuymed,longbuyhigh,state,stpl
0,2019-04-03 09:00:00,19.7475,24.679281,18.247179,22.486269,22.446346,61.922141,50.880358,1,1,1,buylong,0.394950
1,2019-04-03 11:00:00,19.3053,34.827139,25.162081,23.217566,22.057614,61.803550,51.058875,1,1,1,selllong,-0.047250
2,2019-04-03 13:00:00,19.3778,43.836951,34.595241,23.860475,21.799215,61.688073,51.235772,1,1,1,buylong,0.025250
3,2019-04-03 14:00:00,19.2895,40.722729,37.803931,24.050418,21.718477,61.625637,51.323597,1,1,1,selllong,-0.063050
4,2019-04-04 09:00:00,19.4474,43.912940,41.277722,24.197150,21.638435,61.406292,51.584288,1,1,1,buylong,0.094850
5,2019-04-04 11:00:00,19.3132,52.803921,44.910678,24.423349,21.711956,61.253023,51.755556,1,1,1,selllong,-0.039350
6,2019-04-04 14:00:00,19.3763,53.643746,52.573663,24.036467,21.932488,61.006954,52.008513,1,1,1,buylong,0.023750
7,2019-04-08 09:00:00,20.0001,64.266937,67.639763,22.790971,22.882963,60.147706,52.895325,1,0,0,selllong,NaN
8,2019-04-10 14:00:00,20.0791,83.734096,67.846403,22.852051,22.833477,58.392710,54.409336,1,1,1,buylong,0.401582
9,2019-04-11 10:00:00,19.6646,78.112525,82.130267,23.897042,22.775972,58.021715,54.669704,1,1,0,selllong,-0.012918


Index ,Trade, Index sin comission

In [66]:
df = dflongbuysell
df ["index_sc"] = 100. 
df ["trade"] = 0.
df ["index"] = 100. *(1-comission) 
for i in range(1, len(df)):       

                 

    if  df.loc[i, "state"] == "selllong" :
        df.loc[i, "index_sc"] = (((df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"])+1)* df.loc[i-1,"index_sc"]
        df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
        df.loc[i, "trade"] = (df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"]
        
    if  df.loc[i, "state"] == "buylong" :        
        df.loc[i, "index_sc"] =  df.loc[i-1, "index_sc"]
        df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)

display (df.head(10))

,datetime,close,khora,dhora,kdia,ddia,ksem,dsem,longbuylow,longbuymed,longbuyhigh,state,stpl,index_sc,trade,index
0,2019-04-03 09:00:00,19.7475,24.679281,18.247179,22.486269,22.446346,61.922141,50.880358,1,1,1,buylong,0.394950,100.000000,0.000000,99.650000
1,2019-04-03 11:00:00,19.3053,34.827139,25.162081,23.217566,22.057614,61.803550,51.058875,1,1,1,selllong,-0.047250,97.760729,-0.022393,97.418567
2,2019-04-03 13:00:00,19.3778,43.836951,34.595241,23.860475,21.799215,61.688073,51.235772,1,1,1,buylong,0.025250,97.760729,0.000000,97.418567
3,2019-04-03 14:00:00,19.2895,40.722729,37.803931,24.050418,21.718477,61.625637,51.323597,1,1,1,selllong,-0.063050,97.315257,-0.004557,96.974654
4,2019-04-04 09:00:00,19.4474,43.912940,41.277722,24.197150,21.638435,61.406292,51.584288,1,1,1,buylong,0.094850,97.315257,0.000000,96.974654
5,2019-04-04 11:00:00,19.3132,52.803921,44.910678,24.423349,21.711956,61.253023,51.755556,1,1,1,selllong,-0.039350,96.643717,-0.006901,96.305464
6,2019-04-04 14:00:00,19.3763,53.643746,52.573663,24.036467,21.932488,61.006954,52.008513,1,1,1,buylong,0.023750,96.643717,0.000000,96.305464
7,2019-04-08 09:00:00,20.0001,64.266937,67.639763,22.790971,22.882963,60.147706,52.895325,1,0,0,selllong,NaN,99.755062,0.032194,99.405919
8,2019-04-10 14:00:00,20.0791,83.734096,67.846403,22.852051,22.833477,58.392710,54.409336,1,1,1,buylong,0.401582,99.755062,0.000000,99.405919
9,2019-04-11 10:00:00,19.6646,78.112525,82.130267,23.897042,22.775972,58.021715,54.669704,1,1,0,selllong,-0.012918,97.695783,-0.020643,97.353847


stop system

In [69]:
def aplicar_stopsys_dinamico(df, drawmax):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["index"] = pd.to_numeric(df["index"], errors="coerce")

    estado_corrigido = []
    pico_atual = df.loc[0, "index"]

    for i in range(len(df)):
        valor_index = df.loc[i, "index"]
        estado = df.loc[i, "state"]

        # Atualiza pico se houve recuperação
        if valor_index > pico_atual:
            pico_atual = valor_index

        # Calcula drawdown
        if pico_atual > 0:
            drawdown = (valor_index - pico_atual) / pico_atual
        else:
            drawdown = 0

        # Verifica se deve aplicar stopsys
        if estado == "selllong" and drawdown < drawmax:
            estado = "stopsys"
            pico_atual = valor_index  # reinicia ciclo a partir desse ponto

        estado_corrigido.append(estado)

    df["state"] = estado_corrigido
    return df

In [ ]:
df_corrigido = aplicar_stopsys_dinamico(df, drawmax=-0.1)
display(df_corrigido[["datetime", "index", "state"]])

In [73]:
def estatisticas_index(df):
    serie = df["index"].dropna()

    maximo = serie.max()
    minimo = serie.min()
    media = serie.mean()
    desvio = serie.std()
    taxagantot = (maximo - minimo)/minimo

    df["datetime"] = pd.to_datetime(df["datetime"])
    data_inicial = df["datetime"].min()
    data_final = df["datetime"].max()
    dias = (data_final - data_inicial).days
    

    taxagananualprom = (1 + taxagantot) ** (1 / dias * 365) - 1  
    taxalivrerisgoprom = 0.05
    sharpe = taxagananualprom/taxalivrerisgoprom
   
    return pd.Series({
        "Máximo": f"{maximo:.2f}",
        "Mínimo ": f"{minimo:.2f}",
        "Média": f"{media:.2f}",
        "Desvio padrão ": f"{desvio:.2f}",
        "Taxa de ganancia total ": f"{taxagantot * 100:.2f}%",
        "Taxa de ganancia anual prom ": f"{taxagananualprom * 100:.2f}%",
        "coef Sharpe": f"{sharpe:.2f}",
    })

In [75]:
resultado_index = estatisticas_index(df)
print(resultado_index.to_string())

Máximo                          102.00
Mínimo                           57.41
Média                            75.27
Desvio padrão                    13.56
Taxa de ganancia total          77.68%
Taxa de ganancia anual prom     15.81%
coef Sharpe                       3.16


In [77]:
def datas_drawdown_max(df):
    df = df.copy()
    df = df[df["index"].notna()]
    df["datetime"] = pd.to_datetime(df["datetime"])

    # Série com índice acumulado
    acumulado = df["index"]
    pico = acumulado.cummax()
    drawdown = acumulado - pico

    # Índice do drawdown máximo
    idx_vale = drawdown.idxmin()
    idx_pico = (acumulado[:idx_vale]).idxmax()
    idx_trademin = df["trade"].idxmin()


    # Datas correspondentes
    data_pico = df.loc[idx_pico, "datetime"]
    data_vale = df.loc[idx_vale, "datetime"]
    data_trademin = df.loc[idx_trademin, "datetime"]

    # Diferença percentual
    valor_pico = df.loc[idx_pico, "index"]
    valor_vale = df.loc[idx_vale, "index"]
    drawdown_pct = ((valor_vale - valor_pico) / valor_pico) * 100
    trademin = df["trade"].min()
    
    return pd.Series({
        "Data do Pico": data_pico.strftime("%Y-%m-%d %H:%M"),
        "Data do Vale": data_vale.strftime("%Y-%m-%d %H:%M"),
        "Valor do Pico": round(valor_pico, 2),
        "Valor do Vale": round(valor_vale, 2),
        "Drawdown Máximo (%)": f"{drawdown_pct:.2f}%",
        "Màxima perdida por trade": f"{df["trade"].min()*100 :.2f}%",
        "Data do Max Loser Trade": data_trademin.strftime("%Y-%m-%d %H:%M"),
    })

In [79]:
resultado_datas = datas_drawdown_max(df)
print(resultado_datas)

Data do Pico                2019-07-01 14:00
Data do Vale                2022-05-12 09:00
Valor do Pico                          102.0
Valor do Vale                          57.41
Drawdown Máximo (%)                  -43.72%
Màxima perdida por trade              -5.23%
Data do Max Loser Trade     2021-10-07 08:00
dtype: object


In [81]:
def estatisticas_trades(df):
    df = df.copy()
    trades = df["trade"].dropna()

    positivos = trades[trades > 0]
    negativos = trades[trades < 0]

    # Porcentagem de positivos
    porcentagem_pos = (len(positivos) / len(trades)) * 100

    resultado = {
        "Total de Trades": len(trades),
        "Percentual de Trades Positivos (%)": f"{porcentagem_pos:.2f}%",
        "Média dos Trades Positivos": round(positivos.mean(), 6),
        "Desvio Padrão (Trades Positivos)": round(positivos.std(), 6),
        "Média dos Trades Negativos": round(negativos.mean(), 6),
        "Desvio Padrão (Trades Negativos)": round(negativos.std(), 6)
    }

    return pd.Series(resultado)

In [83]:
resumo_trades = estatisticas_trades(df)
print(resumo_trades)

Total de Trades                            146
Percentual de Trades Positivos (%)      12.33%
Média dos Trades Positivos            0.051186
Desvio Padrão (Trades Positivos)      0.058678
Média dos Trades Negativos           -0.020523
Desvio Padrão (Trades Negativos)      0.012081
dtype: object


In [85]:
def ranking_drawdowns_puros(df, coluna="index", top_n=5):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    serie = df[coluna].dropna().reset_index(drop=True)
    datas = df["datetime"].reset_index(drop=True)

    drawdowns = []

    pico_idx = 0
    pico = serie[0]
    vale_idx = None
    valor_vale = None
    max_dd = 0

    for i in range(1, len(serie)):
        if serie[i] > pico:
            # Se recuperou acima do último pico: salvar ciclo anterior
            if vale_idx is not None and max_dd < 0:
                drawdowns.append({
                    "Data Pico": datas[pico_idx],
                    "Valor Pico": pico,
                    "Data Vale": datas[vale_idx],
                    "Valor Vale": valor_vale,
                    "Drawdown (%)": round(max_dd * 100, 2)
                })

            # Novo pico inicia novo ciclo
            pico = serie[i]
            pico_idx = i
            vale_idx = None
            max_dd = 0
        else:
            dd = (serie[i] - pico) / pico
            if dd < max_dd:
                max_dd = dd
                vale_idx = i
                valor_vale = serie[i]

    # Salva último ciclo, se aplicável
    if vale_idx is not None and max_dd < 0:
        drawdowns.append({
            "Data Pico": datas[pico_idx],
            "Valor Pico": pico,
            "Data Vale": datas[vale_idx],
            "Valor Vale": valor_vale,
            "Drawdown (%)": round(max_dd * 100, 2)
        })

    # Retorna os top N
    df_resultado = pd.DataFrame(drawdowns)
    return df_resultado.sort_values("Drawdown (%)").head(top_n).reset_index(drop=True)

In [87]:
ranking_drawdowns_puros(df, coluna="index", top_n=5)


,Data Pico,Valor Pico,Data Vale,Valor Vale,Drawdown (%)
0,2019-07-01 14:00:00,102.004024,2022-05-12 09:00:00,57.409405,-43.72
1,2019-04-03 09:00:00,99.650000,2019-04-04 11:00:00,96.305464,-3.36
2,2019-04-15 14:00:00,100.271202,2019-06-26 13:00:00,97.024371,-3.24


In [89]:
def estatisticas_drawdowns_puros(df, coluna="index"):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    serie = df[coluna].dropna().reset_index(drop=True)
    datas = df["datetime"].reset_index(drop=True)

    drawdowns = []

    pico_idx = 0
    pico = serie[0]
    vale_idx = None
    valor_vale = None
    max_dd = 0

    for i in range(1, len(serie)):
        if serie[i] > pico:
            if vale_idx is not None and max_dd < 0:
                drawdowns.append(max_dd * 100)  # salva como porcentagem
            # novo ciclo
            pico = serie[i]
            pico_idx = i
            max_dd = 0
            vale_idx = None
        else:
            dd = (serie[i] - pico) / pico
            if dd < max_dd:
                max_dd = dd
                vale_idx = i
                valor_vale = serie[i]

    # salva último ciclo, se houver
    if vale_idx is not None and max_dd < 0:
        drawdowns.append(max_dd * 100)

    # série com drawdowns reais
    serie_dd = pd.Series(drawdowns)

    estatisticas = {
        "Total de Drawdowns": len(drawdowns),
        "Média dos Drawdowns (%)": round(serie_dd.mean(), 2),
        "Desvio Padrão (%)": round(serie_dd.std(), 2),
        "Drawdown Máximo (%)": round(serie_dd.min(), 2),
        "Drawdown Mínimo (%)": round(serie_dd.max(), 2)
    }

    return pd.Series(estatisticas)

In [91]:
resumo_dd_puros = estatisticas_drawdowns_puros(df)
print(resumo_dd_puros)

Total de Drawdowns          3.00
Média dos Drawdowns (%)   -16.77
Desvio Padrão (%)          23.34
Drawdown Máximo (%)       -43.72
Drawdown Mínimo (%)        -3.24
dtype: float64


In [93]:
def estatistica_periodos_estaticos_em_dias(df, coluna="index_sc"):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df[df[coluna].notna()].reset_index(drop=True)

    variacao = df[coluna].diff()
    grupos = (variacao != 0).cumsum()

    agrupado = df.groupby(grupos)
    periodos_estaticos = []

    for _, grupo in agrupado:
        if len(grupo) > 1 and grupo[coluna].nunique() == 1:
            duracao_dias = (grupo["datetime"].iloc[-1] - grupo["datetime"].iloc[0]).days
            periodos_estaticos.append({
                "Valor index": grupo[coluna].iloc[0],
                "Data Início": grupo["datetime"].iloc[0],
                "Data Fim": grupo["datetime"].iloc[-1],
                "Duração (dias)": duracao_dias,
                "Número de Registros": len(grupo)
            })

    df_resultado = pd.DataFrame(periodos_estaticos)

    if df_resultado.empty:
        resumo = {
            "Total de Períodos Estáticos": 0,
            "Duração Média (dias)": 0,
            "Maior Duração (dias)": 0,
            "Data do Maior Período": {"Data Início": None, "Data Fim": None}
        }
    else:
        resumo = {
            "Total de Períodos Estáticos": len(df_resultado),
            "Duração Média (dias)": round(df_resultado["Duração (dias)"].mean(), 2),
            "Maior Duração (dias)": df_resultado["Duração (dias)"].max(),
            "Data do Maior Período": df_resultado.loc[df_resultado["Duração (dias)"].idxmax(), ["Data Início", "Data Fim"]].to_dict()
        }

    return df_resultado, pd.Series(resumo)

In [ ]:
df_periodos, estatisticas = estatistica_periodos_estaticos_em_dias(df)
display(df_periodos)
display(estatisticas)